# 🧬 AlphaFold Structure Download - Azure ML Job Submit
---
이 노트북은 `download_alphafold_structure.py`를 사용하여 AlphaFold PDB 구조를 대규모로 수집하는 작업을 제출합니다.
비동기(Async) 다운로드를 통해 속도를 최적화했습니다.

In [ ]:
!pip install azure-ai-ml azure-identity

In [3]:
from pathlib import Path
from azure.ai.ml import MLClient, command, Input
from azure.ai.ml.entities import Environment
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml.constants import AssetTypes
import json
from datetime import datetime

print("✅ 패키지 import 완료")

✅ 패키지 import 완료


## 🎛️ 작업 설정

In [4]:
# ============================================================================
# 🎯 실험 설정
# ============================================================================
EXPERIMENT_NAME = "AlphaFold_Structure_Download"
DISPLAY_NAME = f"af_download_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

# ============================================================================
# 🖥️ 컴퓨팅 설정
# ============================================================================
CLUSTER_NAME = "FOR-CAFA-6"  # GPU 클러스터 (다운로드 작업에는 과분하지만 요청대로 사용)
INSTANCE_COUNT = 1

# ============================================================================
# 🐍 환경 설정
# ============================================================================
ENVIRONMENT_NAME = "alphafold-download-env"
USE_EXISTING_ENV = False  # conda.yaml 수정 반영을 위해 업데이트 필요

# ============================================================================
# 📂 입출력 설정
# ============================================================================
# 소스 폴더 (이 안에 fasta_accessions.txt와 스크립트가 있어야 함)
SRC_DIR = "src"
ACCESSION_FILE = "fasta_accessions.txt"  # SRC_DIR 내부의 파일명

# 작업자 수 (너무 높으면 차단 위험)
WORKERS = 30

print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Cluster: {CLUSTER_NAME}")

Experiment: AlphaFold_Structure_Download
Cluster: FOR-CAFA-6


## 🔐 Azure ML 연결

In [5]:
print("=" * 70)
print("🔐 Azure ML Client 연결 시작")
print("=" * 70)

try:
    try:
        credential = DefaultAzureCredential()
        credential.get_token("https://management.azure.com/.default")
    except Exception:
        print("🌐 브라우저 인증(Interactive)으로 전환합니다...")
        credential = InteractiveBrowserCredential()

    ml_client = MLClient.from_config(credential=credential)
    ws = ml_client.workspaces.get(ml_client.workspace_name)

    print(f"✅ 연결 성공!")
    print(f"   - Workspace: {ws.name}")
    print(f"   - Resource Group: {ws.resource_group}")
    print(f"   - Subscription: {ml_client.subscription_id}")

except Exception as e:
    print(f"❌ 연결 실패: {e}")

🔐 Azure ML Client 연결 시작


DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credentials:
	EnvironmentCredential: EnvironmentCredential authentication unavailable. Environment variables are not fully configured.
Visit https://aka.ms/azsdk/python/identity/environmentcredential/troubleshoot to troubleshoot this issue.
	WorkloadIdentityCredential: WorkloadIdentityCredential authentication unavailable. The workload options are not fully configured. See the troubleshooting guide for more information: https://aka.ms/azsdk/python/identity/workloadidentitycredential/troubleshoot. Missing required arguments: 'tenant_id', 'client_id', 'token_file_path'.
	ManagedIdentityCredential: ManagedIdentityCredential authentication unavailable, no response from the IMDS endpoint.
	SharedTokenCacheCredential: SharedTokenCacheCredential authentication unavailable. No accounts were found in the cache.
	VisualStudioCodeCredential: VisualStudioCodeCredential requires the 'azure-identity-broker' pa

🌐 브라우저 인증(Interactive)으로 전환합니다...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


✅ 연결 성공!
   - Workspace: REDACTED_AZURE_RESOURCE_GROUP-ml
   - Resource Group: REDACTED_AZURE_RESOURCE_GROUP
   - Subscription: REDACTED_AZURE_SUBSCRIPTION_ID


## 🐍 환경 설정 (aiohttp, tqdm 포함)

In [ ]:
env_name = ENVIRONMENT_NAME

if USE_EXISTING_ENV:
    try:
        env = ml_client.environments.get(name=env_name, label="latest")
        print(f"✅ 기존 환경 사용: {env_name} (version: {env.version})")
    except Exception as e:
        print(f"⚠️ 기존 환경 찾을 수 없음: {e}")
        USE_EXISTING_ENV = False

if not USE_EXISTING_ENV:
    conda_yaml_path = Path("env/conda.yaml")
    if not conda_yaml_path.exists():
        raise FileNotFoundError(f"env/conda.yaml 파일을 찾을 수 없습니다.")
    
    env = Environment(
        name=env_name,
        description="CAFA 6 - PyTorch + Async IO",
        conda_file=str(conda_yaml_path),
        image="mcr.microsoft.com/azureml/openmpi4.1.0-cuda11.8-cudnn8-ubuntu22.04:latest",
    )
    env = ml_client.environments.create_or_update(env)
    print(f"✅ 환경 생성/업데이트 완료: {env_name} (version: {env.version})")

✅ 환경 생성/업데이트 완료: alphafold-download-env (version: 1)


## 🚀 Job 생성 및 제출

In [7]:
src_folder = Path(SRC_DIR)

# Command 생성
# src 폴더 전체가 업로드 되므로, ACCESSION_FILE은 현재 디렉토리에 위치하게 됨
command_str = f"""
python download_alphafold_structure.py \
  --accession_list {ACCESSION_FILE} \
  --output_dir ./outputs/structures_af \
  --workers {WORKERS}
""".strip()

job = command(
    code=str(src_folder),
    command=command_str,
    environment=f"{env.name}:{env.version}",
    compute=CLUSTER_NAME,
    instance_count=INSTANCE_COUNT,
    experiment_name=EXPERIMENT_NAME,
    display_name=DISPLAY_NAME,
    description=f"AlphaFold Download (Async, Workers={WORKERS})",
)

print("📤 Job 제출 중...")
try:
    returned_job = ml_client.jobs.create_or_update(job)
    print(f"✅ Job 제출 완료: {returned_job.name}")
    print(f"🔗 URL: {returned_job.studio_url}")
except Exception as e:
    print(f"❌ Job 제출 실패: {e}")

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


📤 Job 제출 중...


Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (1.82 MBs): 100%|##########| 1819978/1819978 [00:00<00:00, 9036171.73it/s]




✅ Job 제출 완료: upbeat_rice_4pygc0zpsq
🔗 URL: https://ml.azure.com/runs/upbeat_rice_4pygc0zpsq?wsid=/subscriptions/REDACTED_AZURE_SUBSCRIPTION_ID/resourcegroups/REDACTED_AZURE_RESOURCE_GROUP/workspaces/REDACTED_AZURE_RESOURCE_GROUP-ml&tid=5fb256f0-fbf2-40d2-81d5-bac1b32c419d
